# Parameters

In [ ]:
# ==========================================
# CONFIGURATION CENTRALE DES PARAMÈTRES
# ==========================================
cfg = {
    'I': 1,           # Carbon Input
    'd': 0.00067,     # Decomposition rate
    'm': 0.02,        # Mortality rate
    'e': 0.6,         # CUE
    'gamma': 1.0,     # Recycling
    'X0': 100.0,      # Initial SOC
    'Y0': 1.0,        # Initial Biomass
    'dt': 0.01,       # Time step
    'n_steps': 500000,# Number of time steps
    'noise_std': 1e-3 # Noise
}

# Model

In [ ]:
# MODEL EQUATIONS

import sympy as sp

# --- 1. Symbol definition ---
X, Y = sp.symbols('X Y')
I, d, m, e = sp.symbols('I d m e')
alpha, beta, gamma = sp.symbols('alpha beta gamma')

# --- 2. Generalized model definition ---
# dX/dt = I - d*X*Y^alpha + gamma*m*Y^beta 
f_gen = I - d * X * Y**alpha + gamma * m * Y**beta
# dY/dt = e*d*X*Y^alpha - m*Y^beta
g_gen = e * d * X * Y**alpha - m * Y**beta

print("GENERALIZED CAUSAL SYSTEM STRUCTURE")
print("-" * 40)
display(sp.Eq(sp.Symbol('dX/dt'), f_gen))
display(sp.Eq(sp.Symbol('dY/dt'), g_gen))
print("-" * 40)

# --- 3. Printing specific model configurations ---
configs = [
    {"name": "LINEAR KINETICS", "alpha": 0, "beta": 1},
    {"name": "MULTIPLICATIVE KINETICS", "alpha": 1, "beta": 1},
    {"name": "DENSITY DEPENDENT MORTALITY", "alpha": 1, "beta": 2}
]

for conf in configs:
    print(f"\nSYSTEM EQUATIONS: {conf['name']}")
    print(f"(Parameters: alpha={conf['alpha']}, beta={conf['beta']})")
    
    # Substituting alpha and beta values into the generalized equations
    f_spec = f_gen.subs({alpha: conf['alpha'], beta: conf['beta']})
    g_spec = g_gen.subs({alpha: conf['alpha'], beta: conf['beta']})
    
    display(sp.Eq(sp.Symbol('dX/dt'), f_spec))
    display(sp.Eq(sp.Symbol('dY/dt'), g_spec))

# Pick model

In [ ]:
# current_conf = {"name": "LINEAR KINETICS", "alpha": 0, "beta": 1}
# current_conf = {"name": "MULTIPLICATIVE KINETICS", "alpha": 1, "beta": 1}## 
current_conf = {"name": "DENSITY DEPENDENT MORTALITY", "alpha": 1, "beta": 2}

# Compartimentation and Mass balance

In [ ]:
# --- COMPARTMENTATION & MASS BALANCE TEST (3 models) ---
# Structural flux-pairing test: condition 3 with α_k ∈ [0,1] and Σα_k ≤ 1
# implies (a) compartmental matrix sign structure and (b) mass balance.

import sympy as sp

param_values = {sp.Symbol('I'): cfg['I'], d: cfg['d'], m: cfg['m'],
                e: cfg['e'], gamma: cfg['gamma']}


def test_flux_pairing(eqs, state_vars, param_values):
    flux_table = {}
    for i, eq in enumerate(eqs):
        for term in sp.Add.make_args(sp.expand(eq)):
            coeff, shape = sp.S(1), sp.S(1)
            for factor in sp.Mul.make_args(term):
                if any(v in factor.free_symbols for v in state_vars):
                    shape *= factor
                else:
                    coeff *= factor
            if shape == 1:
                continue  # external input, not a flux
            flux_table.setdefault(shape, []).append((i, coeff))

    print(f"\nDetected {len(flux_table)} distinct flux shape(s):")
    all_ok = True

    for shape, occ in flux_table.items():
        occ_num = [(i, float(c.subs(param_values))) for i, c in occ]
        negs = [(i, c) for i, c in occ_num if c < 0]
        poss = [(i, c) for i, c in occ_num if c > 0]

        print(f"\n  Flux shape: {shape}")
        for i, c in occ_num:
            role = "SOURCE (−)" if c < 0 else "DESTINATION (+)"
            print(f"    eq {i}: coeff = {c:+.4g}   {role}")

        if len(negs) != 1:
            print(f"    [FAIL] Expected exactly 1 source, found {len(negs)} "
                  f"(orphan or duplicated flux)")
            all_ok = False
            continue

        out_mag = -negs[0][1]
        fracs   = [(i, c / out_mag) for i, c in poss]

        for i, a in fracs:
            ok = (0 <= a <= 1)
            print(f"    α (dest eq {i}) = {a:.4g}   {'[OK]' if ok else '[FAIL]'}")
            if not ok:
                all_ok = False

        sum_a    = sum(a for _, a in fracs)
        outside  = 1 - sum_a
        ok_total = (sum_a <= 1 + 1e-12)
        print(f"    Σ α_k = {sum_a:.4g}   |   fraction to outside = {outside:.4g}   "
              f"{'[OK]' if ok_total else '[FAIL]'}")
        if not ok_total:
            all_ok = False

    return all_ok


# ==================================================================
# LOOP OVER THE 3 MODELS (configs already defined in MODEL EQUATIONS cell)
# ==================================================================
results = {}

for conf in configs:
    print("\n" + "="*60)
    print(f"MODEL: {conf['name']}  (alpha={conf['alpha']}, beta={conf['beta']})")
    print("="*60)

    f_spec = f_gen.subs({alpha: conf['alpha'], beta: conf['beta']})
    g_spec = g_gen.subs({alpha: conf['alpha'], beta: conf['beta']})

    print("System equations:")
    sp.pprint(sp.Eq(sp.Symbol('dX/dt'), f_spec))
    sp.pprint(sp.Eq(sp.Symbol('dY/dt'), g_spec))

    results[conf['name']] = test_flux_pairing([f_spec, g_spec], [X, Y], param_values)

# ==================================================================
print("\n" + "="*60)
print("SUMMARY (Compartmented & mass balanced)")
print("="*60)
for name, ok in results.items():
    print(f"  {name:32} : {ok}")

# Sensitivity analysis

In [ ]:
# --- SENSITIVITY ANALYSIS: 3 MODELS in (e, d) plane ---

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

I_sym = sp.Symbol('I')
fixed = {I_sym: cfg['I'], m: cfg['m'], gamma: cfg['gamma']}

# Steady states derived analytically (all assume e·γ < 1 for positivity)
ss_formulas = {
    "LINEAR KINETICS": (
        I_sym / (d * (1 - e*gamma)),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "MULTIPLICATIVE KINETICS": (
        m / (e * d),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "DENSITY DEPENDENT MORTALITY": (
        sp.sqrt(I_sym * m / (e * (1 - e*gamma))) / d,
        sp.sqrt(I_sym * e / (m * (1 - e*gamma))),
    ),
}

# Grid in (e, d) — same for all models
e_vals = np.linspace(-0.1, 1.5, 250)
d_vals = np.linspace(1e-5, 5e-1, 250)
E, D = np.meshgrid(e_vals, d_vals)
gamma_v = cfg['gamma']


def classify_one_model(conf):
    a, b = conf['alpha'], conf['beta']

    f_spec = f_gen.subs({alpha: a, beta: b})
    g_spec = g_gen.subs({alpha: a, beta: b})

    X_ss_sym, Y_ss_sym = ss_formulas[conf['name']]
    J_sym = sp.Matrix([f_spec, g_spec]).jacobian([X, Y])
    J_at_ss_sym = J_sym.subs({X: X_ss_sym, Y: Y_ss_sym})

    X_ss_f = sp.lambdify((e, d), X_ss_sym.subs(fixed), 'numpy')
    Y_ss_f = sp.lambdify((e, d), Y_ss_sym.subs(fixed), 'numpy')
    J_funcs = [[sp.lambdify((e, d), J_at_ss_sym[i, j].subs(fixed), 'numpy')
                for j in range(2)] for i in range(2)]

    with np.errstate(all='ignore'):
        Xs  = np.broadcast_to(X_ss_f(E, D), E.shape).astype(complex)
        Ys  = np.broadcast_to(Y_ss_f(E, D), E.shape).astype(complex)
        J01 = np.broadcast_to(J_funcs[0][1](E, D), E.shape).astype(complex)
        J10 = np.broadcast_to(J_funcs[1][0](E, D), E.shape).astype(complex)

    pos_ss = ((Xs.real > 0) & (Ys.real > 0)
              & np.isfinite(Xs.real) & np.isfinite(Ys.real)
              & (np.abs(Xs.imag) < 1e-9) & (np.abs(Ys.imag) < 1e-9))

    comp_mb = (E >= 0) & (E <= 1) & (0 <= gamma_v <= 1)

    J01r, J10r = J01.real, J10.real
    tol = 1e-12
    sys_type = np.full(E.shape, 'mixed', dtype=object)
    sys_type[(J01r > tol)        & (J10r > tol)]         = 'cooperative'
    sys_type[(J01r < -tol)       & (J10r < -tol)]        = 'competitive'
    sys_type[((J01r > tol) & (J10r < -tol)) | ((J01r < -tol) & (J10r > tol))] = 'predator-prey'
    sys_type[(np.abs(J01r) < tol) & (np.abs(J10r) < tol)] = 'neutral'
    sys_type[(np.abs(J01r) < tol) & (J10r > tol)]        = 'commensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r > tol)]        = 'commensalism'
    sys_type[(np.abs(J01r) < tol) & (J10r < -tol)]       = 'amensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r < -tol)]       = 'amensalism'
    sys_type[~pos_ss] = 'no SS'

    label_grid = np.empty(E.shape, dtype=object)
    for idx in np.ndindex(E.shape):
        cb = "comp+MB ✓" if comp_mb[idx] else "comp+MB ✗"
        ss = "+SS" if pos_ss[idx] else "−SS"
        label_grid[idx] = f"{cb} | {ss} | {sys_type[idx]}"

    return label_grid


def plot_one_model(conf, label_grid):
    unique_labels = sorted(set(label_grid.flat))
    label_to_int  = {lab: i for i, lab in enumerate(unique_labels)}
    int_grid      = np.vectorize(label_to_int.get)(label_grid)

    fig, ax = plt.subplots(figsize=(14, 6.5))
    cmap = plt.get_cmap('tab10', max(len(unique_labels), 3))
    ax.pcolormesh(e_vals, d_vals, int_grid, cmap=cmap,
                  vmin=-0.5, vmax=len(unique_labels)-0.5, shading='auto')

    ax.set_xlabel('CUE (e)', fontsize=18)
    ax.set_ylabel('Decomposition rate (d)', fontsize=18)
    ax.set_title(f"{conf['name']}  (α={conf['alpha']}, β={conf['beta']})\n"
                 f"(I={cfg['I']}, m={cfg['m']}, γ={cfg['gamma']})",
                 fontsize=17)
    ax.tick_params(axis='both', labelsize=15)

    handles = [Patch(color=cmap(i), label=lab) for i, lab in enumerate(unique_labels)]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1),
              fontsize=13, title='Region (compart+MB | SS | type)',
              title_fontsize=14)

    ax.axvline(0, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axvline(1, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.scatter([cfg['e']], [cfg['d']], color='white', edgecolors='black',
               s=220, marker='*', zorder=5)
    ax.text(cfg['e']+0.03, cfg['d'], 'default', fontsize=14)

    plt.tight_layout()
    plt.show()

    return unique_labels


# --- PASS 1: classify all 3 models and collect every possible label ---
results = {}
all_labels = set()

for conf in configs:
    try:
        results[conf['name']] = classify_one_model(conf)
        all_labels.update(results[conf['name']].flat)
    except Exception as ex:
        print(f"[ERROR] {conf['name']} failed: {ex}")

unique_labels = sorted(all_labels)
label_to_int  = {lab: i for i, lab in enumerate(unique_labels)}
cmap          = plt.get_cmap('tab10', max(len(unique_labels), 3))

# --- PASS 2: plot all 3 models side-by-side with a single shared legend ---
fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=True)

for ax, conf in zip(axes, configs):
    if conf['name'] not in results:
        ax.set_title(f"{conf['name']} (failed)")
        continue
    label_grid = results[conf['name']]
    int_grid   = np.vectorize(label_to_int.get)(label_grid)

    ax.pcolormesh(e_vals, d_vals, int_grid, cmap=cmap,
                  vmin=-0.5, vmax=len(unique_labels)-0.5, shading='auto')
    ax.set_xlabel('CUE (e)', fontsize=18)
    ax.set_title(f"{conf['name']}\n(α={conf['alpha']}, β={conf['beta']})", fontsize=16)
    ax.tick_params(axis='both', labelsize=15)
    ax.axvline(0, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axvline(1, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.scatter([cfg['e']], [cfg['d']], color='white', edgecolors='black',
               s=220, marker='*', zorder=5)
    ax.text(cfg['e']+0.03, cfg['d'], 'default', fontsize=14)

axes[0].set_ylabel('Decomposition rate (d)', fontsize=18)

# Single shared legend on the right
handles = [Patch(color=cmap(i), label=lab) for i, lab in enumerate(unique_labels)]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
           fontsize=14, title='Region (compart+MB | SS | type)',
           title_fontsize=15)

fig.suptitle(f"Sensitivity analysis — 3 model kinetics in (e, d) plane "
             f"(I={cfg['I']}, m={cfg['m']}, γ={cfg['gamma']})",
             fontsize=18, y=1.03)

plt.tight_layout()
plt.show()

# --- Summary per model ---
for conf in configs:
    if conf['name'] not in results:
        continue
    label_grid = results[conf['name']]
    print(f"\n=== {conf['name']} ===")
    print("Regions found:")
    for lab in unique_labels:
        n = int((label_grid == lab).sum())
        if n > 0:
            print(f"  {lab:55s} → {100*n/label_grid.size:5.1f}%  ({n} cells)")


In [ ]:
# --- SENSITIVITY ANALYSIS: 3 MODELS in (e, m) plane ---

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

I_sym = sp.Symbol('I')
fixed = {I_sym: cfg['I'], d: cfg['d'], gamma: cfg['gamma']}  # now d is fixed, m varies

ss_formulas = {
    "LINEAR KINETICS": (
        I_sym / (d * (1 - e*gamma)),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "MULTIPLICATIVE KINETICS": (
        m / (e * d),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "DENSITY DEPENDENT MORTALITY": (
        sp.sqrt(I_sym * m / (e * (1 - e*gamma))) / d,
        sp.sqrt(I_sym * e / (m * (1 - e*gamma))),
    ),
}

# Grid in (e, m)
e_vals = np.linspace(-0.1, 1.5, 250)
m_vals = np.linspace(1e-5, 0.5, 250)
E, M = np.meshgrid(e_vals, m_vals)
gamma_v = cfg['gamma']


def classify_one_model(conf):
    a, b = conf['alpha'], conf['beta']
    f_spec = f_gen.subs({alpha: a, beta: b})
    g_spec = g_gen.subs({alpha: a, beta: b})

    X_ss_sym, Y_ss_sym = ss_formulas[conf['name']]
    J_sym = sp.Matrix([f_spec, g_spec]).jacobian([X, Y])
    J_at_ss_sym = J_sym.subs({X: X_ss_sym, Y: Y_ss_sym})

    X_ss_f = sp.lambdify((e, m), X_ss_sym.subs(fixed), 'numpy')
    Y_ss_f = sp.lambdify((e, m), Y_ss_sym.subs(fixed), 'numpy')
    J_funcs = [[sp.lambdify((e, m), J_at_ss_sym[i, j].subs(fixed), 'numpy')
                for j in range(2)] for i in range(2)]

    with np.errstate(all='ignore'):
        Xs  = np.broadcast_to(X_ss_f(E, M), E.shape).astype(complex)
        Ys  = np.broadcast_to(Y_ss_f(E, M), E.shape).astype(complex)
        J01 = np.broadcast_to(J_funcs[0][1](E, M), E.shape).astype(complex)
        J10 = np.broadcast_to(J_funcs[1][0](E, M), E.shape).astype(complex)

    pos_ss = ((Xs.real > 0) & (Ys.real > 0)
              & np.isfinite(Xs.real) & np.isfinite(Ys.real)
              & (np.abs(Xs.imag) < 1e-9) & (np.abs(Ys.imag) < 1e-9))

    comp_mb = (E >= 0) & (E <= 1) & (0 <= gamma_v <= 1)

    J01r, J10r = J01.real, J10.real
    tol = 1e-12
    sys_type = np.full(E.shape, 'mixed', dtype=object)
    sys_type[(J01r > tol)        & (J10r > tol)]         = 'cooperative'
    sys_type[(J01r < -tol)       & (J10r < -tol)]        = 'competitive'
    sys_type[((J01r > tol) & (J10r < -tol)) | ((J01r < -tol) & (J10r > tol))] = 'predator-prey'
    sys_type[(np.abs(J01r) < tol) & (np.abs(J10r) < tol)] = 'neutral'
    sys_type[(np.abs(J01r) < tol) & (J10r > tol)]        = 'commensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r > tol)]        = 'commensalism'
    sys_type[(np.abs(J01r) < tol) & (J10r < -tol)]       = 'amensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r < -tol)]       = 'amensalism'
    sys_type[~pos_ss] = 'no SS'

    label_grid = np.empty(E.shape, dtype=object)
    for idx in np.ndindex(E.shape):
        cb = "comp+MB ✓" if comp_mb[idx] else "comp+MB ✗"
        ss = "+SS" if pos_ss[idx] else "−SS"
        label_grid[idx] = f"{cb} | {ss} | {sys_type[idx]}"

    return label_grid


# PASS 1: classify all 3 models
results = {}
all_labels = set()
for conf in configs:
    try:
        results[conf['name']] = classify_one_model(conf)
        all_labels.update(results[conf['name']].flat)
    except Exception as ex:
        print(f"[ERROR] {conf['name']} failed: {ex}")

unique_labels = sorted(all_labels)
label_to_int  = {lab: i for i, lab in enumerate(unique_labels)}
cmap          = plt.get_cmap('tab10', max(len(unique_labels), 3))

# PASS 2: plot
fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=True)

for ax, conf in zip(axes, configs):
    if conf['name'] not in results:
        ax.set_title(f"{conf['name']} (failed)")
        continue
    label_grid = results[conf['name']]
    int_grid   = np.vectorize(label_to_int.get)(label_grid)

    ax.pcolormesh(e_vals, m_vals, int_grid, cmap=cmap,
                  vmin=-0.5, vmax=len(unique_labels)-0.5, shading='auto')
    ax.set_xlabel('CUE (e)', fontsize=18)
    ax.set_title(f"{conf['name']}\n(α={conf['alpha']}, β={conf['beta']})", fontsize=16)
    ax.tick_params(axis='both', labelsize=15)
    ax.axvline(0, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axvline(1, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.scatter([cfg['e']], [cfg['m']], color='white', edgecolors='black',
               s=220, marker='*', zorder=5)
    ax.text(cfg['e']+0.03, cfg['m'], 'default', fontsize=14)

axes[0].set_ylabel('Mortality rate (m)', fontsize=18)

handles = [Patch(color=cmap(i), label=lab) for i, lab in enumerate(unique_labels)]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
           fontsize=14, title='Region (compart+MB | SS | type)',
           title_fontsize=15)

fig.suptitle(f"Sensitivity analysis — 3 model kinetics in (e, m) plane "
             f"(I={cfg['I']}, d={cfg['d']}, γ={cfg['gamma']})",
             fontsize=18, y=1.03)

plt.tight_layout()
plt.show()

for conf in configs:
    if conf['name'] not in results:
        continue
    label_grid = results[conf['name']]
    print(f"\n=== {conf['name']} ===")
    print("Regions found:")
    for lab in unique_labels:
        n = int((label_grid == lab).sum())
        if n > 0:
            print(f"  {lab:55s} → {100*n/label_grid.size:5.1f}%  ({n} cells)")


In [ ]:
# --- SENSITIVITY ANALYSIS: 3 MODELS in (e, γ) plane ---

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

I_sym = sp.Symbol('I')
fixed = {I_sym: cfg['I'], d: cfg['d'], m: cfg['m']}  # d and m fixed, e and γ vary

ss_formulas = {
    "LINEAR KINETICS": (
        I_sym / (d * (1 - e*gamma)),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "MULTIPLICATIVE KINETICS": (
        m / (e * d),
        e * I_sym / (m * (1 - e*gamma)),
    ),
    "DENSITY DEPENDENT MORTALITY": (
        sp.sqrt(I_sym * m / (e * (1 - e*gamma))) / d,
        sp.sqrt(I_sym * e / (m * (1 - e*gamma))),
    ),
}

# Grid in (e, γ)
e_vals     = np.linspace(-0.1, 1.5, 300)
gamma_vals = np.linspace(-0.1, 1.5, 300)
E, G = np.meshgrid(e_vals, gamma_vals)


def classify_one_model(conf):
    a, b = conf['alpha'], conf['beta']
    f_spec = f_gen.subs({alpha: a, beta: b})
    g_spec = g_gen.subs({alpha: a, beta: b})

    X_ss_sym, Y_ss_sym = ss_formulas[conf['name']]
    J_sym = sp.Matrix([f_spec, g_spec]).jacobian([X, Y])
    J_at_ss_sym = J_sym.subs({X: X_ss_sym, Y: Y_ss_sym})

    X_ss_f = sp.lambdify((e, gamma), X_ss_sym.subs(fixed), 'numpy')
    Y_ss_f = sp.lambdify((e, gamma), Y_ss_sym.subs(fixed), 'numpy')
    J_funcs = [[sp.lambdify((e, gamma), J_at_ss_sym[i, j].subs(fixed), 'numpy')
                for j in range(2)] for i in range(2)]

    with np.errstate(all='ignore'):
        Xs  = np.broadcast_to(X_ss_f(E, G), E.shape).astype(complex)
        Ys  = np.broadcast_to(Y_ss_f(E, G), E.shape).astype(complex)
        J01 = np.broadcast_to(J_funcs[0][1](E, G), E.shape).astype(complex)
        J10 = np.broadcast_to(J_funcs[1][0](E, G), E.shape).astype(complex)

    pos_ss = ((Xs.real > 0) & (Ys.real > 0)
              & np.isfinite(Xs.real) & np.isfinite(Ys.real)
              & (np.abs(Xs.imag) < 1e-9) & (np.abs(Ys.imag) < 1e-9))

    # Compart+MB: e ∈ [0,1] AND γ ∈ [0,1]
    comp_mb = (E >= 0) & (E <= 1) & (G >= 0) & (G <= 1)

    J01r, J10r = J01.real, J10.real
    tol = 1e-12
    sys_type = np.full(E.shape, 'mixed', dtype=object)
    sys_type[(J01r > tol)        & (J10r > tol)]         = 'cooperative'
    sys_type[(J01r < -tol)       & (J10r < -tol)]        = 'competitive'
    sys_type[((J01r > tol) & (J10r < -tol)) | ((J01r < -tol) & (J10r > tol))] = 'predator-prey'
    sys_type[(np.abs(J01r) < tol) & (np.abs(J10r) < tol)] = 'neutral'
    sys_type[(np.abs(J01r) < tol) & (J10r > tol)]        = 'commensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r > tol)]        = 'commensalism'
    sys_type[(np.abs(J01r) < tol) & (J10r < -tol)]       = 'amensalism'
    sys_type[(np.abs(J10r) < tol) & (J01r < -tol)]       = 'amensalism'
    sys_type[~pos_ss] = 'no SS'

    label_grid = np.empty(E.shape, dtype=object)
    for idx in np.ndindex(E.shape):
        cb = "comp+MB ✓" if comp_mb[idx] else "comp+MB ✗"
        ss = "+SS" if pos_ss[idx] else "−SS"
        label_grid[idx] = f"{cb} | {ss} | {sys_type[idx]}"

    return label_grid


# PASS 1: classify all 3 models
results = {}
all_labels = set()
for conf in configs:
    try:
        results[conf['name']] = classify_one_model(conf)
        all_labels.update(results[conf['name']].flat)
    except Exception as ex:
        print(f"[ERROR] {conf['name']} failed: {ex}")

unique_labels = sorted(all_labels)
label_to_int  = {lab: i for i, lab in enumerate(unique_labels)}
cmap          = plt.get_cmap('tab10', max(len(unique_labels), 3))

# PASS 2: plot
fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=True)

for ax, conf in zip(axes, configs):
    if conf['name'] not in results:
        ax.set_title(f"{conf['name']} (failed)")
        continue
    label_grid = results[conf['name']]
    int_grid   = np.vectorize(label_to_int.get)(label_grid)

    ax.pcolormesh(e_vals, gamma_vals, int_grid, cmap=cmap,
                  vmin=-0.5, vmax=len(unique_labels)-0.5, shading='auto')
    ax.set_xlabel('CUE (e)', fontsize=18)
    ax.set_title(f"{conf['name']}\n(α={conf['alpha']}, β={conf['beta']})", fontsize=16)
    ax.tick_params(axis='both', labelsize=15)
    ax.axvline(0, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axvline(1, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axhline(0, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.axhline(1, color='k', ls='--', alpha=0.3, lw=0.8)
    ax.scatter([cfg['e']], [cfg['gamma']], color='white', edgecolors='black',
               s=220, marker='*', zorder=5)
    ax.text(cfg['e']+0.03, cfg['gamma'], 'default', fontsize=14)

axes[0].set_ylabel('Recycling (γ)', fontsize=18)

handles = [Patch(color=cmap(i), label=lab) for i, lab in enumerate(unique_labels)]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5),
           fontsize=14, title='Region (compart+MB | SS | type)',
           title_fontsize=15)

fig.suptitle(f"Sensitivity analysis — 3 model kinetics in (e, γ) plane "
             f"(I={cfg['I']}, d={cfg['d']}, m={cfg['m']})",
             fontsize=18, y=1.03)

plt.tight_layout()
plt.show()

for conf in configs:
    if conf['name'] not in results:
        continue
    label_grid = results[conf['name']]
    print(f"\n=== {conf['name']} ===")
    print("Regions found:")
    for lab in unique_labels:
        n = int((label_grid == lab).sum())
        if n > 0:
            print(f"  {lab:55s} → {100*n/label_grid.size:5.1f}%  ({n} cells)")


# Jacobian

In [ ]:
# --- 4. STEADY STATE & STRUCTURAL GAIN ANALYSIS ---

print(f"\n>>> ANALYZING: {current_conf['name']} (Steady State & Gain)")
print("-" * 60)

# A. Substitution of alpha and beta
f_spec = f_gen.subs({alpha: current_conf['alpha'], beta: current_conf['beta']})
g_spec = g_gen.subs({alpha: current_conf['alpha'], beta: current_conf['beta']})

# B. Solving for Steady States
ss_solutions = sp.solve([f_spec, g_spec], (X, Y))

# --- AJOUT : Normalisation pour gérer les formats dict et list ---
if isinstance(ss_solutions, dict):
    ss_list = [(ss_solutions[X], ss_solutions[Y])]
elif isinstance(ss_solutions, list) and len(ss_solutions) > 0 and isinstance(ss_solutions[0], dict):
    ss_list = [(s[X], s[Y]) for s in ss_solutions]
else:
    ss_list = ss_solutions

num_solutions = len(ss_list)
print(f"[WARNING] {num_solutions} steady state(s) found.") if num_solutions > 1 else print("Unique steady state found.")

# --- C. Numerical Evaluation & Selection ---
params_values = {I: cfg['I'], d: cfg['d'], m: cfg['m'], e: cfg['e'], gamma: cfg['gamma']}
valid_solutions = []

for i, sol in enumerate(ss_list): # <--- Changer ss_solutions par ss_list
    x_val = float(sp.re(sol[0].subs(params_values)))
    y_val = float(sp.re(sol[1].subs(params_values)))
    
    is_positive = (x_val > 0 and y_val > 0)
    if is_positive:
        valid_solutions.append(sol)
    
    print(f"Solution #{i+1}: X_ss = {x_val:.4f}, Y_ss = {y_val:.4f} | Positive: {is_positive}")

# --- D. Selection Logic for Gain Calculation ---
if len(valid_solutions) == 0:
    print("\n[ERROR] No positive steady state found. Gain calculation aborted.")
elif len(valid_solutions) > 1:
    print("\n[ERROR] Several steady states are positive. Automatic gain calculation aborted.")
else:
    # We have exactly one positive solution
    X_val_ss, Y_val_ss = valid_solutions[0]
    print(f"\nProceeding with the unique positive steady state.")

    # --- Définition de la Jacobienne pour le modèle courant ---
    F = sp.Matrix([f_spec, g_spec])
    vars_mat = sp.Matrix([X, Y])
    J_sym = F.jacobian(vars_mat)
    
    # Évaluation au Steady State sélectionné
    J_at_ss = J_sym.subs({X: X_val_ss, Y: Y_val_ss})
    
    print("\n2. Jacobian Matrix at Steady State (s_ij):")
    display(J_at_ss)

    # --- E. Structural Gain Calculation ---
    j11, j12 = J_at_ss[0,0], J_at_ss[0,1]
    j21, j22 = J_at_ss[1,0], J_at_ss[1,1]
    
    denominator = j11 * j22
    if denominator == 0:
        print("\n[ERROR] Gain calculation aborted because denominator is zero.")
    else:
        symbolic_gain = sp.simplify((j12 * j21) / denominator)
        
        print("\n3. Symbolic Gain Expression:")
        # display() renders the expression with Greek letters and proper math formatting
        display(sp.Eq(sp.Symbol('G'), symbolic_gain))

        # --- F. Final Numerical Gain ---
        params_values = {I: cfg['I'], d: cfg['d'], m: cfg['m'], e: cfg['e'], gamma: cfg['gamma']}
        numerical_gain_expr = symbolic_gain.subs(params_values)
        final_gain = float(sp.re(numerical_gain_expr))
        
        print(f"\n4. Numerical Gain at Steady State: {final_gain:.4f}")

# Jacobian terms

In [ ]:
# --- JACOBIAN TERMS AT STEADY STATE (3 MODELS) ---

import sympy as sp
import pandas as pd

# Symbols (same as in your "Model" cell)
X, Y = sp.symbols('X Y')
I, d, m, e = sp.symbols('I d m e')
alpha, beta, gamma = sp.symbols('alpha beta gamma')

f_gen = I - d * X * Y**alpha + gamma * m * Y**beta
g_gen = e * d * X * Y**alpha - m * Y**beta

configs = [
    {"name": "LINEAR KINETICS",            "alpha": 0, "beta": 1},
    {"name": "MULTIPLICATIVE KINETICS",    "alpha": 1, "beta": 1},
    {"name": "DENSITY DEPENDENT MORTALITY","alpha": 1, "beta": 2},
]

params_values = {I: cfg['I'], d: cfg['d'], m: cfg['m'],
                 e: cfg['e'], gamma: cfg['gamma']}

results = []

for conf in configs:
    print("\n" + "="*60)
    print(f">>> {conf['name']} (alpha={conf['alpha']}, beta={conf['beta']})")
    print("="*60)

    # 1. Specific equations
    f_spec = f_gen.subs({alpha: conf['alpha'], beta: conf['beta']})
    g_spec = g_gen.subs({alpha: conf['alpha'], beta: conf['beta']})

    # 2. Steady states
    ss_solutions = sp.solve([f_spec, g_spec], (X, Y))
    if isinstance(ss_solutions, dict):
        ss_list = [(ss_solutions[X], ss_solutions[Y])]
    elif isinstance(ss_solutions, list) and len(ss_solutions) > 0 and isinstance(ss_solutions[0], dict):
        ss_list = [(s[X], s[Y]) for s in ss_solutions]
    else:
        ss_list = ss_solutions

    # 3. Pick the unique positive steady state
    valid = []
    for sol in ss_list:
        x_val = float(sp.re(sol[0].subs(params_values)))
        y_val = float(sp.re(sol[1].subs(params_values)))
        if x_val > 0 and y_val > 0:
            valid.append((sol, x_val, y_val))

    if len(valid) != 1:
        print(f"[SKIP] {len(valid)} positive steady states found.")
        continue

    (X_ss_sym, Y_ss_sym), X_ss, Y_ss = valid[0]
    print(f"Steady state: X_ss = {X_ss:.4f} | Y_ss = {Y_ss:.4f}")

    # 4. Symbolic Jacobian
    J_sym = sp.Matrix([f_spec, g_spec]).jacobian(sp.Matrix([X, Y]))
    J_at_ss_sym = sp.simplify(J_sym.subs({X: X_ss_sym, Y: Y_ss_sym}))

    print("\nSymbolic Jacobian at steady state:")
    display(J_at_ss_sym)

    # 5. Numerical Jacobian (the 4 terms)
    J_num = J_at_ss_sym.subs(params_values)
    j11 = float(sp.re(J_num[0, 0]))
    j12 = float(sp.re(J_num[0, 1]))
    j21 = float(sp.re(J_num[1, 0]))
    j22 = float(sp.re(J_num[1, 1]))

    print("\nNumerical Jacobian terms at steady state:")
    print(f"  j11 (df/dX) = {j11: .6e}")
    print(f"  j12 (df/dY) = {j12: .6e}")
    print(f"  j21 (dg/dX) = {j21: .6e}")
    print(f"  j22 (dg/dY) = {j22: .6e}")

    # 6. Structural gain
    gain = (j12 * j21) / (j11 * j22) if j11 * j22 != 0 else float('nan')
    print(f"  Gain G = (j12*j21)/(j11*j22) = {gain: .6f}")

    results.append({
        'Model':  conf['name'],
        'X_ss':   X_ss,
        'Y_ss':   Y_ss,
        'j11':    j11,
        'j12':    j12,
        'j21':    j21,
        'j22':    j22,
        'Gain':   gain,
    })

# --- Synthesis table ---
df_jac = pd.DataFrame(results).set_index('Model')
print("\n" + "="*60)
print("SYNTHESIS — Jacobian at steady state")
print("="*60)
print(df_jac.to_string(float_format=lambda v: f"{v: .4e}"))


# Generate 10 time series

In [ ]:
# --- GÉNÉRATION DE 10 SÉRIES TEMPORELLES ---

import numpy as np

all_simulations = []
n_simulations = 10

print(f">>> GENERATING {n_simulations} TIME SERIES: {current_conf['name']}")

for i in range(n_simulations):
    X = np.zeros(cfg['n_steps'])
    Y = np.zeros(cfg['n_steps'])
    X[0] = cfg['X0']
    Y[0] = cfg['Y0']
    
    a, b = current_conf['alpha'], current_conf['beta']

    for t in range(1, cfg['n_steps']):
        y_safe = max(Y[t-1], 1e-10)
        flux_decomp = cfg['d'] * X[t-1] * (y_safe**a)
        flux_mortality = cfg['m'] * (y_safe**b)
        
        dX_val = (cfg['I'] - flux_decomp + cfg['gamma'] * flux_mortality) * cfg['dt']
        dY_val = (cfg['e'] * flux_decomp - flux_mortality) * cfg['dt']
        
        X[t] = X[t-1] + dX_val + np.random.normal(0, cfg['noise_std'])
        Y[t] = Y[t-1] + dY_val + np.random.normal(0, cfg['noise_std'])
    
    all_simulations.append({'X': X, 'Y': Y})
    print(f" Simulation {i+1}/{n_simulations} terminée.")

In [ ]:
# Plot des time series

import matplotlib.pyplot as plt
import numpy as np

# 1. Extraction des données des simulations existantes
X_stack = np.array([s['X'] for s in all_simulations])
Y_stack = np.array([s['Y'] for s in all_simulations])
t_axis = np.arange(cfg['n_steps']) * cfg['dt']

# 2. Plot
fig, ax1 = plt.subplots(figsize=(10, 5))

# SOC (Bleu)
ax1.plot(t_axis, X_stack.mean(axis=0), color='tab:blue', lw=2, label='Moyenne SOC')
ax1.fill_between(t_axis, X_stack.mean(axis=0) - X_stack.std(axis=0), 
                 X_stack.mean(axis=0) + X_stack.std(axis=0), color='tab:blue', alpha=0.2)
ax1.set_ylabel('SOC (X)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Biomasse (Vert) sur axe secondaire
ax2 = ax1.twinx()
ax2.plot(t_axis, Y_stack.mean(axis=0), color='tab:green', lw=2, label='Moyenne Biomasse')
ax2.fill_between(t_axis, Y_stack.mean(axis=0) - Y_stack.std(axis=0), 
                 Y_stack.mean(axis=0) + Y_stack.std(axis=0), color='tab:green', alpha=0.2)
ax2.set_ylabel('Biomasse (Y)', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

plt.title(f"Synthèse des 10 séries temporelles - {current_conf['name']}")
ax1.grid(True, alpha=0.2)
fig.tight_layout()
plt.show()

# Granger for 10 time series

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ==================================================================================
# CONFIGURATION ET FONCTION DE GRANGER (MARINAZZO ET AL. 2008)
# ==================================================================================

# Paramètres d'analyse
N_MAX_GRANGER = 3000
M_LISTE_CV = [1, 2, 3, 4, 5]
P_LISTE_EXPLORATION = [1, 2, 3, 4]

def marinazzo_2008_steady_state(X_in, Y_in, t_X, t_Y, m_list, p_list, alpha=0.05):
    """
    Analyse la causalité et retourne le meilleur degré polynomial pour chaque direction.
    """
    def create_lagged_matrix(series, m):
        return np.column_stack([series[m-1-i : len(series)-i] for i in range(m)])

    # --- ÉTAPE 1 : SÉLECTION DE M ---
    mse_total = []
    for m in m_list:
        errors_m = []
        for series, evol in [(X_in, t_X), (Y_in, t_Y)]:
            X_p = create_lagged_matrix(series, m)
            t_adj = evol[m-1:] 
            split = int(len(t_adj) * 0.8)
            tr_p, val_p = X_p[:split], X_p[split:]
            tr_t, val_t = t_adj[:split], t_adj[split:]
            mu, std = np.mean(tr_p, axis=0), np.std(tr_p, axis=0) + 1e-10
            tr_n, val_n = (tr_p - mu)/std, (val_p - mu)/std
            inv_k = np.linalg.pinv(np.dot(tr_n.T, tr_n) + np.eye(m) * 1e-6)
            beta = inv_k @ tr_n.T @ (tr_t - np.mean(tr_t))
            pred_val = (val_n @ beta) + np.mean(tr_t)
            errors_m.extend((val_t - pred_val)**2)
        mse_total.append(np.mean(errors_m))

    best_m = m_list[np.argmin(mse_total)]
    
    # --- ÉTAPE 2 : ÉVALUATION DE LA CAUSALITÉ (p) ---
    m = best_m
    t_X_adj, t_Y_adj = t_X[m-1:], t_Y[m-1:]
    X_p = create_lagged_matrix(X_in, m)
    Y_p = create_lagged_matrix(Y_in, m)
    N = len(t_X_adj)
    J = np.eye(N) - np.ones((N, N)) / N
    
    history = {"Y->X": [], "X->Y": []}
    results_summary = {}

    for p in p_list:
        Z_raw = np.column_stack([X_p, Y_p])
        Z_n = (Z_raw - np.mean(Z_raw, axis=0)) / (np.std(Z_raw, axis=0) + 1e-10)
        K_glob = J @ ((1 + np.dot(Z_n, Z_n.T))**p) @ J
        ev, evec = np.linalg.eigh(K_glob)
        K_filt = (evec[:, ev >= ev.max()*1e-6] * ev[ev >= ev.max()*1e-6]) @ evec[:, ev >= ev.max()*1e-6].T
        
        for source_data, target_past, t_evol, lab in [(Z_n[:, m:], Z_n[:, :m], t_X_adj, "Y->X"), 
                                                      (Z_n[:, :m], Z_n[:, m:], t_Y_adj, "X->Y")]:
            y_c = t_evol - np.mean(t_evol)
            y_norm = y_c / (np.linalg.norm(y_c) + 1e-10)
            K_t = J @ ((1 + np.dot(target_past, target_past.T))**p) @ J
            ev_h, evec_h = np.linalg.eigh(K_t)
            P = evec_h[:, ev_h >= ev_h.max()*1e-6] @ evec_h[:, ev_h >= ev_h.max()*1e-6].T
            eps = 1 - np.dot(P @ y_norm, P @ y_norm) 
            K_tilde = K_filt - P@K_filt - K_filt@P + P@K_filt@P 
            ev_kt, evec_kt = np.linalg.eigh(K_tilde)
            keep_kt = ev_kt >= (ev_kt.max() * 1e-6 if ev_kt.max() > 0 else 1e-6)
            m_sig = np.sum(keep_kt)
            bonf = alpha / m_sig if m_sig > 0 else alpha 
            r_sq = [stats.pearsonr(evec_kt[:, i], y_norm)[0]**2 for i in range(len(ev_kt)) 
                    if keep_kt[i] and stats.pearsonr(evec_kt[:, i], y_norm)[1] < bonf]
            
            df = np.sum(r_sq) / eps if eps > 1e-12 else 0.0 
            history[lab].append(df)

    # Extraction des conclusions
    for lab in ["Y->X", "X->Y"]:
        scores = history[lab]
        if not scores or max(scores) == 0:
            results_summary[lab] = "None"
        else:
            best_p = p_list[np.argmax(scores)]
            results_summary[lab] = f"Linear" if best_p == 1 else f"Poly(d={best_p})"
            
    return results_summary

# ==================================================================================
# BOUCLE D'ANALYSE SUR LES 10 SÉRIES TEMPORELLES
# ==================================================================================

all_conclusions = []

print(f"--- STARTING AUTOMATED ANALYSIS (10 RUNS) ---")

for idx, sim in enumerate(all_simulations):
    # 1. Extraction des données (3000 derniers points)
    X_ss = sim['X'][-N_MAX_GRANGER:]
    Y_ss = sim['Y'][-N_MAX_GRANGER:]
    
    # 2. Préparation des deltas (dX/dt)
    target_x = np.diff(X_ss)
    target_y = np.diff(Y_ss)
    X_in_ss = X_ss[:-1]
    Y_in_ss = Y_ss[:-1]
    
    # 3. Lancement de Granger
    res = marinazzo_2008_steady_state(X_in_ss, Y_in_ss, target_x, target_y, M_LISTE_CV, P_LISTE_EXPLORATION)
    
    # 4. Stockage
    res['Sim_ID'] = idx + 1
    all_conclusions.append(res)
    print(f"Simulation {idx+1}/10: X->Y = {res['X->Y']} | Y->X = {res['Y->X']}")

# ==================================================================================
# RÉSUMÉ FINAL
# ==================================================================================
# Table
df_summary = pd.DataFrame(all_conclusions).set_index('Sim_ID')
print("\n" + "="*40)
print("FINAL SYNTHESIS TABLE")
print("="*40)
print(df_summary)
print("="*40)

# Stats
def get_stats_string(series):
    counts = series.value_counts()
    total = len(series)
    
    # On récupère les catégories et on les trie par fréquence décroissante
    # counts.index contient les noms (None, Linear, etc.)
    # counts.values contient le nombre d'occurrences
    sorted_categories = counts.index.tolist()
    sorted_categories.sort(key=lambda x: counts[x], reverse=True)
    
    parts = []
    for cat in sorted_categories:
        perc = (counts[cat] / total) * 100
        
        # Formatage du nom pour coller à ton souhait
        if cat == "None":
            label = "no effect"
        elif cat == "Linear":
            label = "linear effect"
        else:
            # Transforme Poly(d=3) en polynomial degree 3 effect
            label = cat.lower().replace("poly(d=", "polynomial degree ").replace(")", "") + " effect"
            
        parts.append(f"{perc:.0f}% {label}")
    
    return ", ".join(parts)

# --- Affichage final ---
print("\n" + "="*60)
print("SYNTHESIS GRANGER (ON 10 RUNS)")
print("="*60)

print(f"X->Y (SOC->B): {get_stats_string(df_summary['X->Y'])}")
print(f"Y->X (B->SOC): {get_stats_string(df_summary['Y->X'])}")
print("="*60)

# MLP for 10 time series

In [ ]:
# MLP - 1000 époques, step size 20 - retrait des tests statistiques

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

# --- 1. ARCHITECTURE DU MODÈLE ---
class CausalMLP(nn.Module):
    def __init__(self):
        super(CausalMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(), 
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

# --- 2. LE MOTEUR DE CALCUL (ENGINE) ---
def run_tesch_causal_engine(in_data, out_data, n_iterations=3, return_local_gains=True, dt_val=0.01):
    """Version optimisée et autonome pour le calcul du gain temporel."""
    gains_list = []
    
    for s in range(n_iterations):
        sc_in, sc_out = StandardScaler(), StandardScaler()
        in_s = torch.tensor(sc_in.fit_transform(in_data), dtype=torch.float32, requires_grad=True)
        out_s = torch.tensor(sc_out.fit_transform(out_data), dtype=torch.float32)

        model = CausalMLP()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        
        # Entraînement rapide (1000 époques suffisent pour capturer la tendance)
        model.train()
        for epoch in range(2000):
            optimizer.zero_grad()
            output = model(in_s)
            loss = criterion(output, out_s)
            loss.backward()
            optimizer.step()

        model.eval()
        
        # Calcul vectorisé du Jacobien
        preds = model(in_s)
        grads_dX = torch.autograd.grad(preds[:, 0].sum(), in_s, retain_graph=True)[0]
        grads_dY = torch.autograd.grad(preds[:, 1].sum(), in_s)[0]
        
        # Conversion physique
        q11 = (grads_dX[:, 0].detach().numpy() * (sc_out.scale_[0] / sc_in.scale_[0])) / dt_val
        q12 = (grads_dX[:, 1].detach().numpy() * (sc_out.scale_[0] / sc_in.scale_[1])) / dt_val
        q21 = (grads_dY[:, 0].detach().numpy() * (sc_out.scale_[1] / sc_in.scale_[0])) / dt_val
        q22 = (grads_dY[:, 1].detach().numpy() * (sc_out.scale_[1] / sc_in.scale_[1])) / dt_val
        
        local_gains = (q12 * q21) / ((q11 * q22) + 1e-10)
        gains_list.append(local_gains)
        
    return np.array(gains_list)

# --- 3. ANALYSE ET PLOT DES 10 SIMULATIONS ---
def plot_causal_average_with_variance(simulations_list, step_size=20):
    all_gains_matrix = []
    
    print(f"Analyse des {len(simulations_list)} simulations pour calcul de la moyenne...")
    
    for i, sim in enumerate(simulations_list):
        # Préparation des données
        X_sub = sim['X'][::step_size]
        Y_sub = sim['Y'][::step_size]
        eff_dt = cfg['dt'] * step_size
        
        dX = np.diff(X_sub)
        dY = np.diff(Y_sub)
        in_raw = np.column_stack([X_sub[:-1], Y_sub[:-1]])
        out_raw = np.column_stack([dX, dY])
        
        # Calcul du gain pour cette simulation
        # On utilise le moteur Tesch autonome défini précédemment
        gain_curve = run_tesch_causal_engine(in_raw, out_raw, n_iterations=1, dt_val=eff_dt)[0]
        all_gains_matrix.append(gain_curve)
        print(f"  > Simulation {i+1} traitée.")

    # Convertir en matrice numpy pour les calculs statistiques
    # Format : (n_simulations, n_time_steps)
    all_gains_matrix = np.array(all_gains_matrix)
    
    # Calcul des statistiques temporelles
    mean_gain = np.mean(all_gains_matrix, axis=0)
    std_gain = np.std(all_gains_matrix, axis=0)
    
    # --- GÉNÉRATION DU PLOT ---
    plt.figure(figsize=(12, 6))
    time_axis = np.arange(len(mean_gain)) * step_size
    
    # 1. Tracer la ligne moyenne (Bleu foncé)
    plt.plot(time_axis, mean_gain, color='#004488', linewidth=2, label='Gain Moyen (10 Sims)')
    
    # 2. Tracer la plage de variance (Bleu clair)
    # On utilise 1 écart-type (std_gain) pour la plage
    plt.fill_between(time_axis, 
                     mean_gain - std_gain, 
                     mean_gain + std_gain, 
                     color='#6699CC', alpha=0.3, label='Écart-type ($\pm \sigma$)')

    # Ajout d'une ligne horizontale à 0 pour référence
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)

    # Formatage
    plt.title(f"Synthèse du Gain Causal (Moyenne $\pm$ Variance) - {current_conf['name']}", fontsize=14)
    plt.xlabel("Temps (Steps)", fontsize=12)
    plt.ylabel("Gain $G(t)$", fontsize=12)
    plt.grid(True, alpha=0.2)
    plt.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# --- LANCEMENT ---
plot_causal_average_with_variance(all_simulations, step_size=20)

# Jacobian for 10 time series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import lambdify

def plot_jacobian_average_with_variance(simulations_list, step_size=20):
    # --- 1. DÉFINITION SYMBOLIQUE ---
    X_s, Y_s = sp.symbols('X Y')
    I, d, m, e, gamma = sp.symbols('I d m e gamma')
    alpha_s, beta_s = sp.symbols('alpha beta')

    y_safe = sp.Max(Y_s, 1e-10)
    flux_decomp = d * X_s * (Y_s**alpha_s)
    flux_mortality = m * (Y_s**beta_s)

    f_eq = I - flux_decomp + gamma * flux_mortality
    g_eq = e * flux_decomp - flux_mortality

    F_mat = sp.Matrix([f_eq, g_eq])
    Vars_mat = sp.Matrix([X_s, Y_s])
    J_sym = F_mat.jacobian(Vars_mat)

    # --- 2. PRÉPARATION DES FONCTIONS NUMÉRIQUES ---
    params_vals = {
        I: cfg['I'], d: cfg['d'], m: cfg['m'], 
        e: cfg['e'], gamma: cfg['gamma'],
        alpha_s: current_conf['alpha'], 
        beta_s: current_conf['beta']
    }

    j11_func = lambdify((X_s, Y_s), J_sym[0,0].subs(params_vals), 'numpy')
    j12_func = lambdify((X_s, Y_s), J_sym[0,1].subs(params_vals), 'numpy')
    j21_func = lambdify((X_s, Y_s), J_sym[1,0].subs(params_vals), 'numpy')
    j22_func = lambdify((X_s, Y_s), J_sym[1,1].subs(params_vals), 'numpy')

    all_jacobian_gains = []

    print(f"Calcul de la moyenne Jacobienne sur {len(simulations_list)} simulations...")

    for i, sim in enumerate(simulations_list):
        # On utilise le même sous-échantillonnage que le MLP pour la cohérence
        Xt = sim['X'][::step_size]
        Yt = sim['Y'][::step_size]
        
        # Évaluation de la Jacobienne
        v11 = j11_func(Xt, Yt)
        v12 = j12_func(Xt, Yt)
        v21 = j21_func(Xt, Yt)
        v22 = j22_func(Xt, Yt)
        
        # Correction pour les modèles linéaires (scalaires -> vecteurs)
        if np.isscalar(v11):
            v11, v12, v21, v22 = [np.full_like(Xt, v) for v in [v11, v12, v21, v22]]
        
        # Calcul du gain structurel
        with np.errstate(divide='ignore', invalid='ignore'):
            gain_t = (v12 * v21) / (v11 * v22 + 1e-10)
            gain_t = np.nan_to_num(gain_t, nan=0.0)
            
        all_jacobian_gains.append(gain_t)

    # --- 3. STATISTIQUES ET PLOT ---
    all_jacobian_gains = np.array(all_jacobian_gains)
    mean_jac = np.mean(all_jacobian_gains, axis=0)
    std_jac = np.std(all_jacobian_gains, axis=0)
    
    time_axis = np.arange(len(mean_jac)) * step_size * cfg['dt']

    plt.figure(figsize=(12, 6))
    
    # Ligne moyenne (Rouge foncé pour distinguer du MLP bleu)
    plt.plot(time_axis, mean_jac, color='#880000', linewidth=2, label='Gain Jacobien Moyen')
    
    # Plage de variance (Rouge clair)
    plt.fill_between(time_axis, 
                     mean_jac - std_jac, 
                     mean_jac + std_jac, 
                     color='#CC6666', alpha=0.3, label='Écart-type ($\pm \sigma$)')

    # Ligne de référence à 0
    plt.axhline(0, color='black', linestyle='--', alpha=0.5)

    plt.title(f"Synthèse du Gain Structurel (Jacobienne) - {current_conf['name']}", fontsize=14)
    plt.xlabel("Temps (Units)", fontsize=12)
    plt.ylabel("Gain $G(t)$", fontsize=12)
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.show()

# --- LANCEMENT ---
plot_jacobian_average_with_variance(all_simulations, step_size=20)

# Jacobian vs MLP - 10 times series

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import lambdify

# --- 1. ARCHITECTURE ET MOTEUR CAUSAL (AUTONOME) ---
class CausalMLP(nn.Module):
    def __init__(self):
        super(CausalMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(), 
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

def run_tesch_causal_engine(in_data, out_data, n_iterations=1, dt_val=0.01):
    """Calcule le gain via la sensibilité du réseau de neurones."""
    gains_list = []
    for s in range(n_iterations):
        sc_in, sc_out = StandardScaler(), StandardScaler()
        in_s = torch.tensor(sc_in.fit_transform(in_data), dtype=torch.float32, requires_grad=True)
        out_s = torch.tensor(sc_out.fit_transform(out_data), dtype=torch.float32)

        model = CausalMLP()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        model.train()
        for epoch in range(2000): # Convergence profonde pour réduire la variance 
            optimizer.zero_grad()
            output = model(in_s)
            loss = nn.MSELoss()(output, out_s)
            loss.backward()
            optimizer.step()

        model.eval()
        preds = model(in_s)
        grads_dX = torch.autograd.grad(preds[:, 0].sum(), in_s, retain_graph=True)[0]
        grads_dY = torch.autograd.grad(preds[:, 1].sum(), in_s)[0]
        
        q11 = (grads_dX[:, 0].detach().numpy() * (sc_out.scale_[0] / sc_in.scale_[0])) / dt_val
        q12 = (grads_dX[:, 1].detach().numpy() * (sc_out.scale_[0] / sc_in.scale_[1])) / dt_val
        q21 = (grads_dY[:, 0].detach().numpy() * (sc_out.scale_[1] / sc_in.scale_[0])) / dt_val
        q22 = (grads_dY[:, 1].detach().numpy() * (sc_out.scale_[1] / sc_in.scale_[1])) / dt_val
        
        gains_list.append((q12 * q21) / ((q11 * q22) + 1e-10))
    return np.array(gains_list)

# --- 2. FONCTION DE COMPARAISON MLP vs JACOBIENNE ---
def plot_combined_causal_analysis(simulations_list, step_size=20):
    all_mlp_gains, all_jac_gains = [], []
    
    # Définition symbolique de la Jacobienne
    X_s, Y_s = sp.symbols('X Y')
    I, d, m, e, gamma, a_s, b_s = sp.symbols('I d m e gamma alpha beta')
    flux_decomp = d * X_s * (Y_s**a_s)
    flux_mortality = m * (Y_s**b_s)
    f_eq = I - flux_decomp + gamma * flux_mortality
    g_eq = e * flux_decomp - flux_mortality
    J_sym = sp.Matrix([f_eq, g_eq]).jacobian(sp.Matrix([X_s, Y_s]))

    p_vals = {I: cfg['I'], d: cfg['d'], m: cfg['m'], e: cfg['e'], 
              gamma: cfg['gamma'], a_s: current_conf['alpha'], b_s: current_conf['beta']}

    j_funcs = [lambdify((X_s, Y_s), J_sym[i, j].subs(p_vals), 'numpy') for i in range(2) for j in range(2)]

    print(f"Analyse croisée sur {len(simulations_list)} simulations...")

    for i, sim in enumerate(simulations_list):
        X_sub, Y_sub = sim['X'][::step_size], sim['Y'][::step_size]
        eff_dt = cfg['dt'] * step_size
        
        # A. Gain MLP
        in_r = np.column_stack([X_sub[:-1], Y_sub[:-1]])
        out_r = np.column_stack([np.diff(X_sub), np.diff(Y_sub)])
        all_mlp_gains.append(run_tesch_causal_engine(in_r, out_r, dt_val=eff_dt)[0])
        
        # B. Gain Jacobien
        v = [f(X_sub, Y_sub) for f in j_funcs]
        if np.isscalar(v[0]): v = [np.full_like(X_sub, val) for val in v]
        
        with np.errstate(divide='ignore', invalid='ignore'):
            jac_c = (v[1][:-1] * v[2][:-1]) / (v[0][:-1] * v[3][:-1] + 1e-10)
            all_jac_gains.append(np.nan_to_num(jac_c, nan=0.0))
        print(f"  > Simulation {i+1} terminée.")

    # Statistiques
    m_mlp, s_mlp = np.mean(all_mlp_gains, axis=0), np.std(all_mlp_gains, axis=0)
    m_jac, s_jac = np.mean(all_jac_gains, axis=0), np.std(all_jac_gains, axis=0)
    t_axis = np.arange(len(m_mlp)) * step_size * cfg['dt']

    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(t_axis, m_mlp, color='#004488', label='Gain MLP', lw=2)
    plt.fill_between(t_axis, m_mlp - s_mlp, m_mlp + s_mlp, color='#004488', alpha=0.15)
    plt.plot(t_axis, m_jac, color='#880000', label='Gain Jacobien', lw=2, ls='--')
    plt.fill_between(t_axis, m_jac - s_jac, m_jac + s_jac, color='#880000', alpha=0.15)
    
    # --- AJOUT DES LIMITES ET FORMATAGE ---
    plt.ylim(-1, 0.5) # Force les limites de l'axe Y
    plt.axhline(0, color='black', lw=1, alpha=0.5) # Ligne de référence à 0
    
    plt.title(f"Comparaison Causalité IA vs Jacobienne - {current_conf['name']}")
    plt.xlabel("Temps (Units)"); plt.ylabel("Gain $G(t)$"); plt.legend(); plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

# --- 3. LANCEMENT ---
plot_combined_causal_analysis(all_simulations, step_size=20)